<a href="https://colab.research.google.com/github/anastasiakalyashova/python-ai-AnastasiaKalyashova/blob/main/week3d_violin_rocks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══════════════════════════════════════════════════════
#  ЯЧЕЙКА 0. Подготовка данных (из week2b_read_csv.ipynb)
#  Запускать первой в каждом ноутбуке задания 3
# ═══════════════════════════════════════════════════════

# --- Параметры (изменять здесь) ----------------------
RADIUS_KM     = 300   # радиус соседства гор (для week3a)
TOP_N_ROCKS   = 10    # сколько топ-пород использовать
TOP_N_COMPLEX = 20    # сколько самых «сложных» гор брать
# -----------------------------------------------------

import os, pandas as pd, numpy as np
from itertools import combinations

# 1. Клонируем репозиторий (если ещё нет)
repo = "python-ai-AnastasiaKalyashova"
repo_path = f"/content/{repo}"
if not os.path.exists(repo_path):
    !git clone -q https://github.com/anastasiakalyashova/python-ai-AnastasiaKalyashova.git
if os.getcwd() != repo_path:
    %cd {repo_path}

# 2. Читаем CSV
file_path = None
for root, dirs, files in os.walk("."):
    if "mountains.csv" in files:
        file_path = os.path.join(root, "mountains.csv")
        break
df = pd.read_csv(file_path)

# 3. Переименование столбцов
if "mountainLabel" in df.columns:
    df = df.rename(columns={
        "mountain":          "URL",
        "mountainLabel":     "mountain",
        "rockMaterialLabel": "rockMaterial",
        "elevationMeters":   "elevation",
    })

# 4. Нормализуем породы
df["rockMaterial"] = df["rockMaterial"].str.lower().str.strip()

# 🔧 ИСПРАВЛЕНИЕ: заменяем "lutite" на "пелит"
df["rockMaterial"] = df["rockMaterial"].replace("lutite", "пелит")

# 5. Парсим координаты
coords = df["coordinates"].str.extract(r'Point\(([^\s]+)\s+([^\s]+)\)')
df["lon"] = pd.to_numeric(coords[0], errors="coerce")
df["lat"] = pd.to_numeric(coords[1], errors="coerce")

# 6. df_unique — по одной строке на гору
df_unique = (
    df.groupby("URL")
    .agg(
        mountain   = ("mountain",     "first"),
        lon        = ("lon",          "first"),
        lat        = ("lat",          "first"),
        elevation  = ("elevation",    "first"),
        rock_count = ("rockMaterial", "nunique"),
        rocks      = ("rockMaterial", lambda x: list(x.unique())),
    )
    .reset_index()
)

# 7. df_clean — только физически возможные высоты
df_clean = df_unique[
    (df_unique.elevation >= 0) &
    (df_unique.elevation <= 8849)
].copy()

# 8. Топ пород по частоте (по df_clean)
top_rocks = (
    df[df["URL"].isin(df_clean["URL"])]
    ["rockMaterial"].value_counts()
    .head(TOP_N_ROCKS).index.tolist()
)

# 9. Co-occurrence матрица пород
pairs = []
for rocks in df_clean["rocks"]:
    clean = [r for r in rocks if r in top_rocks]
    pairs += list(combinations(sorted(set(clean)), 2))
cooc = (pd.DataFrame(pairs, columns=["r1", "r2"])
        .value_counts()
        .reset_index(name="count"))

print(f"✅ Длинный формат:    {len(df)} строк")
print(f"✅ Уникальных гор:    {len(df_unique)}")
print(f"✅ df_clean:          {len(df_clean)} гор (0–8849 м)")
print(f"✅ Топ-{TOP_N_ROCKS} пород:    {top_rocks}")
print(f"✅ Пар co-occurrence: {len(cooc)}")

/content/python-ai-AnastasiaKalyashova
✅ Длинный формат:    4431 строк
✅ Уникальных гор:    2915
✅ df_clean:          2914 гор (0–8849 м)
✅ Топ-10 пород:    ['известняк', 'песчаник', 'гранит', 'мергель', 'конгломерат', 'пелит', 'доломит', 'андезит', 'осадочная горная порода', 'базальт']
✅ Пар co-occurrence: 15


In [ ]:
# ═══════════════════════════════════════════════════════
# week3d_violin_rocks.ipynb — Голос каждой породы
# ═══════════════════════════════════════════════════════

import plotly.graph_objects as go
import plotly.io as pio
import numpy as np

pio.templates.default = "plotly_white"

print("📊 Строим violin plot для топ-пород...")
print(f"   Топ-{TOP_N_ROCKS} пород: {top_rocks}")

# 1. Подготавливаем данные
df_violin = df[
    df["rockMaterial"].isin(top_rocks) &
    df["URL"].isin(df_clean["URL"])
].copy()

rock_count_dict = df_clean.set_index("URL")["rock_count"].to_dict()
df_violin["rock_count"] = df_violin["URL"].map(rock_count_dict)
df_violin["complexity"] = df_violin["rock_count"].apply(
    lambda x: "single" if x == 1 else "multi"
)

print(f"   Данных для визуализации: {len(df_violin)} записей")

# 2. Горы выше 5000 м
high_mountains = df_violin[df_violin["elevation"] >= 5000].copy()
print(f"   Гор выше 5000 м: {len(high_mountains)}")

# 3. Строим график
fig = go.Figure()

# Для каждой породы
for i, rock in enumerate(top_rocks):
    rock_data = df_violin[df_violin["rockMaterial"] == rock]

    single_rock = rock_data[rock_data["complexity"] == "single"]["elevation"]
    multi_rock = rock_data[rock_data["complexity"] == "multi"]["elevation"]

    # Левая половина (однопородные)
    if len(single_rock) > 0:
        fig.add_trace(go.Violin(
            y=single_rock,
            x=[i] * len(single_rock),
            legendgroup=rock,
            scalegroup=rock,
            side='negative',
            line_color='#1f77b4',
            fillcolor='rgba(31, 119, 180, 0.5)',
            showlegend=False,
            name=rock,
            hovertemplate='<b>%{text}</b><br>Высота: %{y:.0f} м<br>Тип: однопородная<extra></extra>',
            text=[f"{row['mountain']}<br>({row['rock_count']} порода)"
                  for _, row in rock_data[rock_data["complexity"] == "single"].iterrows()],
        ))

    # Правая половина (многопородные)
    if len(multi_rock) > 0:
        fig.add_trace(go.Violin(
            y=multi_rock,
            x=[i] * len(multi_rock),
            legendgroup=rock,
            scalegroup=rock,
            side='positive',
            line_color='#ff7f0e',
            fillcolor='rgba(255, 127, 14, 0.5)',
            showlegend=(len(single_rock) == 0),
            name=rock,
            hovertemplate='<b>%{text}</b><br>Высота: %{y:.0f} м<br>Тип: многопородная<br>Пород: %{customdata}<extra></extra>',
            customdata=[row['rock_count'] for _, row in rock_data[rock_data["complexity"] == "multi"].iterrows()],
            text=[f"{row['mountain']}"
                  for _, row in rock_data[rock_data["complexity"] == "multi"].iterrows()],
        ))

# 4. Jitter-точки для гор выше 5000 м
np.random.seed(42)
for i, rock in enumerate(top_rocks):
    high_points = high_mountains[high_mountains["rockMaterial"] == rock]
    if len(high_points) > 0:
        jitter_offset = np.random.uniform(-0.3, 0.3, len(high_points))
        jitter_x = [i + offset for offset in jitter_offset]  # ← здесь i (число), а не rock (строка)!

        fig.add_trace(go.Scatter(
            x=jitter_x,
            y=high_points["elevation"],
            mode='markers',
            marker=dict(size=7, color='black', opacity=0.7, symbol='circle', line=dict(width=0.5, color='white')),
            legendgroup=rock,
            showlegend=False,
            hovertemplate='<b>%{text}</b><br>🏔️ Высота: %{y:.0f} м<br>📚 Пород: %{customdata}<extra></extra>',
            text=high_points["mountain"],
            customdata=high_points["rock_count"],
            name=f"{rock}_points"
        ))

# 5. Box plot внутри violins
for i, rock in enumerate(top_rocks):
    rock_data = df_violin[df_violin["rockMaterial"] == rock]["elevation"]
    if len(rock_data) > 0:
        fig.add_trace(go.Box(
            y=rock_data,
            x=[i] * len(rock_data),
            legendgroup=rock,
            boxmean=False,
            boxpoints=False,
            fillcolor='rgba(0,0,0,0)',
            line_color='black',
            marker_color='black',
            line_width=1.5,
            showlegend=False,
            hoverinfo='skip',
            width=0.3,
            name=f"{rock}_box"
        ))

# 6. Настройка внешнего вида
fig.update_layout(
    title=dict(
        text="<b>Голос каждой породы</b><br><sup>Распределение высот: однопородные (слева) vs многопородные (справа)</sup>",
        x=0.5, xanchor='center', font=dict(size=16)
    ),
    xaxis=dict(
        title="<b>Тип горной породы</b>",
        tickangle=-45,
        tickfont=dict(size=11),
        tickmode='array',
        tickvals=list(range(len(top_rocks))),
        ticktext=top_rocks
    ),
    yaxis=dict(
        title="<b>Высота (метры)</b>",
        gridcolor='lightgray', gridwidth=0.5
    ),
    legend=dict(
        title="<b>Легенда</b>",
        yanchor="top", y=0.99, xanchor="left", x=0.01,
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black', borderwidth=1
    ),
    height=600, width=1300, hovermode='closest'
)

# Горизонтальная линия на 5000 м
fig.add_hline(y=5000, line_dash="dash", line_color="red", opacity=0.5,
              annotation_text="5000 м — граница высоких гор", annotation_position="top right")

fig.update_xaxes(showgrid=False, zeroline=False)
fig.update_yaxes(zeroline=True, zerolinewidth=1, zerolinecolor='lightgray')

# 7. Легенда
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                         marker=dict(size=12, color='#1f77b4', symbol='square'),
                         name='🏔️ Однопородные (1 порода)', showlegend=True))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                         marker=dict(size=12, color='#ff7f0e', symbol='square'),
                         name='🗻 Многопородные (≥2 пород)', showlegend=True))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                         marker=dict(size=7, color='black', symbol='circle', line=dict(width=0.5, color='white')),
                         name='⬆️ Горы >5000 м (с названием)', showlegend=True))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                         line=dict(color='black', width=1.5),
                         name='📊 Box plot (медиана, квартили)', showlegend=True))

fig.show()

# 8. Статистика
print("\n" + "="*60)
print("📊 Анализ: однопородные vs многопородные горы")
print("="*60)

stats = []
for rock in top_rocks:
    rock_data = df_violin[df_violin["rockMaterial"] == rock]
    single_data = rock_data[rock_data["complexity"] == "single"]["elevation"]
    multi_data = rock_data[rock_data["complexity"] == "multi"]["elevation"]

    stats.append({
        "Порода": rock,
        "Однопородных": len(single_data),
        "Многопородных": len(multi_data),
        "Медиана (single)": single_data.median() if len(single_data) > 0 else np.nan,
        "Медиана (multi)": multi_data.median() if len(multi_data) > 0 else np.nan,
        "Разница": (multi_data.median() - single_data.median())
                   if len(single_data) > 0 and len(multi_data) > 0 else np.nan
    })

stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False, float_format="%.0f", na_rep="—"))

print("\n" + "="*60)
print("💡 Однопородные и многопородные горы одной породы —")
print("   они живут на разных высотах?")
print("="*60)

stats_with_diff = stats_df[~stats_df["Разница"].isna()].copy()
if len(stats_with_diff) > 0:
    max_diff = stats_with_diff.loc[stats_with_diff["Разница"].idxmax()]
    min_diff = stats_with_diff.loc[stats_with_diff["Разница"].idxmin()]

    if max_diff["Разница"] > 0:
        print(f"\n📈 Многопородные ВЫШЕ: {max_diff['Порода']} (+{max_diff['Разница']:.0f} м)")
        print(f"   single: {max_diff['Медиана (single)']:.0f} м → multi: {max_diff['Медиана (multi)']:.0f} м")

    if min_diff["Разница"] < 0:
        print(f"\n📉 Многопородные НИЖЕ: {min_diff['Порода']} ({min_diff['Разница']:.0f} м)")
        print(f"   single: {min_diff['Медиана (single)']:.0f} м → multi: {min_diff['Медиана (multi)']:.0f} м")

📊 Строим violin plot для топ-пород...
   Топ-10 пород: ['известняк', 'песчаник', 'гранит', 'мергель', 'конгломерат', 'пелит', 'доломит', 'андезит', 'осадочная горная порода', 'базальт']
   Данных для визуализации: 3213 записей
   Гор выше 5000 м: 42



📊 Анализ: однопородные vs многопородные горы
                 Порода  Однопородных  Многопородных  Медиана (single)  Медиана (multi)  Разница
              известняк           475            363              1552              792     -760
               песчаник            64            456              1005              880     -125
                 гранит           336             21              2683             2885      202
                мергель            18            282              1113              711     -402
            конгломерат           147            147              1107              895     -212
                  пелит             1            230              1767             2006      240
                доломит            79            104              2126              793    -1333
                андезит           165              9              2376             1637     -739
осадочная горная порода           159              3              2014           